# Fink/LSST — Reload and Display Saved Light Curves (Tags × DDFs)

This notebook reads the Parquet files saved by `07_fink_tags_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per Fink tag.

**Expected data** in `data_FINK_BLOCK_LC_07/`:
- `{tag}_fp.parquet`  — forced-photometry light curves
- `{tag}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics
- `tag_ranking.csv`   — variability ranking by tag
- `visit_summary_src.csv` / `visit_summary_fp.csv` — visit-level summaries

No API call is made in this notebook.

## Differences from notebook 02 (block replot)

| Aspect | NB 02 (blocks) | NB 08 (tags) |
|--------|---------------|-------------|
| Source data dir | `data_FINK_BLOCK_LC_01` | `data_FINK_BLOCK_LC_07` |
| `lc_cache` structure | `lc_cache[group][oid]` | `lc_cache[oid]` with `group`/`tag`/`field` keys |
| Group key | `classify_object()` category | Fink tag name directly |
| `field` column | yes | yes (DDF name, identical semantics) |
| `tag` column | absent | present (= group) |
| Flatness filtering | `CALIBRATION_EXCLUDE` set | all tags kept (no calibration filter) |
| Extra plots | — | per-DDF flatness boxplot, tag ranking table |


## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import collections
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07"
DIR_DATA = f"data_{NB_TAG}"  # input:  written by notebook 07
DIR_FIGS = f"figs_{NB_TAG}_08"  # output: figures specific to this replot notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per tag
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ─────────────────────────────────────────────────────────────
# Choisir quels tags afficher :
#   'all'  -> tous les tags présents sur disque (comportement par défaut)
#   list   -> sous-ensemble de tags, e.g. ['in_tns', 'hostless_candidate']
PLOT_MODE = "all"  # <── changer ici : 'all' | liste de tags

# ── Known Fink tag descriptions (used for display) ───────────────────────────
FINK_TAGS = {
    "extragalactic_lt20mag_candidate": "Rising, bright (mag < 20), extragalactic candidates",
    "extragalactic_new_candidate": "New (< 48 h first detection) and potentially extragalactic",
    "hostless_candidate": "Hostless alerts according to ELEPHANT (arXiv:2404.18165)",
    "in_tns": "Alerts with a known counterpart in TNS (AT or confirmed)",
    "sn_near_galaxy_candidate": "Alerts matching a galaxy catalog and consistent with SNe",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

## 3. Discover available tags from Parquet files on disk

In [ ]:
# Auto-discover available tags from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def tag_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


tags_fp = {tag_from_path(p, "_fp.parquet"): p for p in fp_files}
tags_src = {tag_from_path(p, "_src.parquet"): p for p in src_files}
all_tags = sorted(set(tags_fp) | set(tags_src))

# Apply PLOT_MODE filter
if PLOT_MODE == "all":
    selected_tags = list(all_tags)
elif isinstance(PLOT_MODE, list):
    selected_tags = [t for t in all_tags if t in PLOT_MODE]
else:
    selected_tags = list(all_tags)
    print(f"WARNING: unknown PLOT_MODE={PLOT_MODE!r}, defaulting to 'all'")

print(f"Tags on disk: {len(all_tags)},  sélectionnés (PLOT_MODE={PLOT_MODE!r}): {len(selected_tags)}")
print()
for t in all_tags:
    has_fp = "yes" if t in tags_fp else "no "
    has_src = "yes" if t in tags_src else "no "
    sel = "<<" if t in selected_tags else "  "
    desc = FINK_TAGS.get(t, "(unknown tag)")
    print(f"  {sel} {t:50s}  fp={has_fp}  src={has_src}  |  {desc}")

## 4. Load Parquet files — reconstruct `lc_cache`

The cache structure mirrors notebook 07:
```python
lc_cache[oid] = {
    'fp'   : DataFrame,   # forced photometry
    'src'  : DataFrame,   # diaSources
    'group': str,         # = tag name
    'tag'  : str,         # = tag name
    'field': str,         # DDF name
}
```

In [ ]:
lc_cache = {}  # oid (str) → {fp, src, group, tag, field}

for tag in all_tags:
    for lc_type, path_dict in (("fp", tags_fp), ("src", tags_src)):
        if tag not in path_dict:
            continue
        df = pd.read_parquet(path_dict[tag])

        # Drop residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag/mag_err if absent
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # The Parquet files written by NB07 contain 'diaObjectId', 'group', 'tag', 'field' columns.
        oid_col = "diaObjectId" if "diaObjectId" in df.columns else "r:diaObjectId"

        for oid, sub in df.groupby(oid_col):
            oid = str(oid)
            if oid not in lc_cache:
                # Retrieve metadata from the first row (same for all rows of an oid)
                row0 = sub.iloc[0]
                lc_cache[oid] = {
                    "fp": pd.DataFrame(),
                    "src": pd.DataFrame(),
                    "group": str(row0.get("group", tag)),
                    "tag": str(row0.get("tag", tag)),
                    "field": str(row0.get("field", "unknown")),
                }
            lc_cache[oid][lc_type] = sub.reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total objects loaded: {len(lc_cache)}")
print()

# Build group_oids index: tag -> list of oids
all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Objects loaded per tag   [DDF breakdown]:")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    fields_in_group = collections.Counter(lc_cache[o]["field"] for o in group_oids[g])
    field_str = "  ".join(f"{f}:{n}" for f, n in sorted(fields_in_group.items()))
    n_pts = sum(len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"]) for o in group_oids[g])
    print(f"  {g:50s}  {len(group_oids[g]):3d} objects  {n_pts:6d} pts   {field_str}")

## 5. Load flatness metrics and tag ranking

In [ ]:
# ── Flatness metrics ──────────────────────────────────────────────────────────
csv_path = os.path.join(DIR_DATA, "flatness_metrics.csv")
if os.path.exists(csv_path):
    df_flat = pd.read_csv(csv_path)
    print(f"flatness_metrics.csv loaded: {len(df_flat)} rows")
    print("\nMedian flatness by tag:")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
    print("\nMedian flatness by DDF:")
    if "field" in df_flat.columns:
        print(df_flat.groupby("field")[["n_pts", "rms_var"]].median().round(4))
else:
    df_flat = pd.DataFrame()
    print("flatness_metrics.csv not found — flatness plots will be skipped.")

# ── Tag ranking ───────────────────────────────────────────────────────────────
ranking_path = os.path.join(DIR_DATA, "tag_ranking.csv")
if os.path.exists(ranking_path):
    df_ranking = pd.read_csv(ranking_path)
    print(f"\ntag_ranking.csv loaded: {len(df_ranking)} rows")
else:
    df_ranking = pd.DataFrame()
    print("\ntag_ranking.csv not found.")

## 6. Flatness boxplot per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle(
        "Flux variability by Fink tag (all DDFs combined) — lower = flatter",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_tag")
    plt.show()

## 6b. Flatness boxplot per DDF (all tags combined)

In [ ]:
if df_flat.empty or "field" not in df_flat.columns:
    print("No flatness / field data available.")
else:
    fields_present = sorted(df_flat["field"].dropna().unique())
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_field = [df_b[df_b["field"] == f]["rms_var"].dropna().values for f in fields_present]
        bp = ax.boxplot(
            data_per_field, labels=fields_present, patch_artist=True, notch=False, showfliers=True
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by DDF (all tags combined)", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01b_flatness_boxplot_by_field")
    plt.show()

## 7. Light-curve plotting functions

In [ ]:
def rank_oids(oid_list, nc=NC):
    """Return the top-nc object IDs, ranked by total number of data points."""
    scored = [(o, len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def plot_lc_grid(oid_list, group, mode="flux", nc=NC):
    """
    Plot a (n_objects × n_bands) grid of light curves for one Fink tag.

    Parameters
    ----------
    oid_list : list of str   — object IDs belonging to this tag
    group    : str           — tag name (used for title and file name)
    mode     : str           — 'flux' (nJy) or 'mag' (AB magnitude)
    nc       : int           — maximum number of objects to plot
    """
    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for tag {group}.")
        return

    fig, axes = plt.subplots(
        n_obj,
        len(BANDS),
        figsize=(2.8 * len(BANDS), 2.6 * n_obj),
        sharex=False,
        sharey=False,
        squeeze=False,
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        field = d.get("field", "")

        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        ax_first_band = None
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")

            if len(df_b) < 3:
                ax.set_visible(False)
                continue
            if ax_first_band is None:
                ax_first_band = ax

            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
            else:
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
            df_b = df_b[mask].reset_index(drop=True)

            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            dt = df_b["r:midpointMjdTai"].values
            dt = dt - dt.min()
            color = BAND_COLORS.get(band, "gray")

            if mode == "flux":
                y, yerr = df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values
            else:
                y, yerr = df_b["mag"].values, df_b["mag_err"].values
                ax.invert_yaxis()

            ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)
            rms = rms_variability(df_b["r:psfFlux"].values)
            ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)

        label_ax = ax_first_band if ax_first_band is not None else axes[row_idx][0]
        field_label = f"  [{field}]" if field else ""
        label_ax.set_ylabel(
            f"{oid}{field_label}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}",
            fontsize=9,
        )

    fig.suptitle(f"Tag: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plotting functions defined.")

## 8. Courbes de lumière en flux

Contrôlé par `PLOT_MODE` (section 1) : `'all'` · liste de tags.

In [ ]:
groups_to_plot = [g for g in selected_tags if g in group_oids and len(group_oids[g]) >= 1]

for group in groups_to_plot:
    n_obj = len(group_oids[group])
    desc = FINK_TAGS.get(group, "")
    print(f"\n=== {group} ({n_obj} objects) — flux ===")
    if desc:
        print(f"    {desc}")
    plot_lc_grid(group_oids[group], group, mode="flux")

## 9. Courbes de lumière en magnitude

Même sélection `PLOT_MODE` que la section 8.

In [ ]:
for group in groups_to_plot:
    n_obj = len(group_oids[group])
    print(f"\n=== {group} ({n_obj} objects) — magnitude ===")
    plot_lc_grid(group_oids[group], group, mode="mag")

## 10. Detailed view — single object inspector

Set `TARGET_TAG` and `TARGET_OID` below to inspect any individual object.

In [ ]:
# ── Choose the tag and object to inspect ──────────────────────────────────────
TARGET_TAG = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(group_oids[TARGET_TAG], nc=5)
TARGET_OID = TOP_OIDS[0]  # ← change here (pick from TOP_OIDS)

print(f"Tag      : {TARGET_TAG}")
print(f"Object   : {TARGET_OID}")
print(f"DDF      : {lc_cache[TARGET_OID]['field']}")
print(f"Top OIDs : {TOP_OIDS}")

# ── Build combined light curve ────────────────────────────────────────────────
d = lc_cache[TARGET_OID]
frames = [df for df in (d["fp"], d["src"]) if not df.empty]
df_obj = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)

print(f"\nData points per band:")
print(df_obj.groupby("r:band").size().rename("n_pts").to_frame())

In [ ]:
# ── Detailed flux + magnitude side-by-side plot per band ──────────────────────
bands_obj = [b for b in BANDS if b in df_obj["r:band"].values]

fig, axes = plt.subplots(len(bands_obj), 2, figsize=(12, 3.2 * len(bands_obj)), squeeze=False)

for row_idx, band in enumerate(bands_obj):
    df_b = df_obj[df_obj["r:band"] == band].copy()
    dt = df_b["r:midpointMjdTai"].values
    dt = dt - dt.min()
    color = BAND_COLORS.get(band, "gray")

    # Flux panel
    ax0 = axes[row_idx][0]
    mask_f = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    ax0.errorbar(
        dt[mask_f],
        df_b["r:psfFlux"].values[mask_f],
        yerr=df_b["r:psfFluxErr"].values[mask_f],
        fmt="o",
        ms=3,
        lw=0.8,
        elinewidth=0.8,
        color=color,
        alpha=0.85,
    )
    rms = rms_variability(df_b["r:psfFlux"].values[mask_f])
    ax0.set_ylabel(f"Flux (nJy) — band {band}", color=color)
    ax0.set_xlabel("Δt (days)")
    ax0.set_title(f"Flux  N={mask_f.sum()}  σ/<f>={rms:.4f}", fontsize=8, color=color)

    # Magnitude panel
    ax1 = axes[row_idx][1]
    if "mag" in df_b.columns and "mag_err" in df_b.columns:
        mask_m = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
        ax1.errorbar(
            dt[mask_m],
            df_b["mag"].values[mask_m],
            yerr=df_b["mag_err"].values[mask_m],
            fmt="o",
            ms=3,
            lw=0.8,
            elinewidth=0.8,
            color=color,
            alpha=0.85,
        )
        ax1.invert_yaxis()
        ax1.set_ylabel(f"AB mag — band {band}", color=color)
        ax1.set_xlabel("Δt (days)")
        ax1.set_title(f"Magnitude  N={mask_m.sum()}", fontsize=8, color=color)
    else:
        ax1.set_visible(False)

field_label = f" | DDF={lc_cache[TARGET_OID]['field']}" if lc_cache[TARGET_OID].get("field") else ""
fig.suptitle(
    f"diaObjectId = {TARGET_OID}  |  tag = {TARGET_TAG}{field_label}",
    fontsize=11,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
savefig(f"03_detail_{TARGET_OID}")
plt.show()

## 11. Summary scatter plot — variability vs mean flux per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"].replace("_", "\n"),
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Fink tag variability summary (objects in DDFs)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("04_tag_summary_scatter")
    plt.show()

    print("\nTop rows sorted by (band, median_rms):")
    display(summary.sort_values(["band", "median_rms"]).head(40))

## 12. Final ranking table — variability by tag

In [ ]:
if not df_ranking.empty:
    print("Ranking by photometric variability (ascending RMS) — all Fink tags in DDFs:")
    print("=" * 110)
    cols = [
        c
        for c in ["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]
        if c in df_ranking.columns
    ]
    print(df_ranking[cols].to_string(index=False, float_format="{:.4f}".format))

    # ── Breakdown by (tag, DDF) ───────────────────────────────────────────────
    if not df_flat.empty and "field" in df_flat.columns:
        print("\nMedian RMS by (tag, DDF):")
        breakdown = (
            df_flat.groupby(["group", "field"])
            .agg(n_obj=("diaObjectId", "nunique"), median_rms=("rms_var", "median"))
            .reset_index()
            .sort_values(["group", "median_rms"])
        )
        display(breakdown)
elif not df_flat.empty:
    # Recompute ranking on the fly from flatness metrics
    print("tag_ranking.csv not found — recomputing from flatness_metrics.csv.")
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    ranking["description"] = ranking["group"].map(FINK_TAGS).fillna("")
    display(ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]])
else:
    print("No ranking data available.")